# CTF vs Baselines: Near-Duplicate Text Detection Demo

This notebook demonstrates Circular Text Fingerprints (CTF) — an ECFP-style iterative neighborhood hashing method — compared against baseline methods (MinHash, character n-grams, word Jaccard, TF-IDF) for detecting near-duplicate questions in the Quora Question Pairs dataset.

**What this demo does:**
1. Loads a curated subset of QQP pairs
2. Splits into train/test (60/40)
3. Tunes similarity thresholds on train data
4. Evaluates on test data and reports precision, recall, F1
5. Visualizes results by method and word-count strata

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import gc
import hashlib
import json
import math
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity
from loguru import logger
import sys

# Suppress verbose logging for demo
logger.remove()
logger.add(sys.stderr, level="WARNING")

In [ ]:
# GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2cec91-circular-text-fingerprints-evaluating-ec/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=5) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    import os
    local_path = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.', 'mini_demo_data.json')
    if os.path.exists(local_path):
        with open(local_path) as f:
            return json.load(f)
    
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

# Load data
print("Loading demo data...")
data = load_data()
print(f"✓ Loaded data with {len(data['datasets'][0]['examples'])} examples")

## Configuration

These parameters control the demo scale. Start minimal for testing, scale up for better results.

In [ ]:
# ── Demo-specific configuration ──
# Start with minimum values for fast iteration; scale up if it works
N_EXAMPLES = 20  # Small demo: use 20 examples. Can scale to 30+ for better results.
BOOTSTRAP_N = 100  # Reduced from 1000 for demo speed. Original: 1000
TFIDF_BATCH = 50  # Batch size for TF-IDF. Original: 2000
THRESHOLD_GRANULARITY = 0.01  # Grid search granularity (0.01 = 100 thresholds)
TRAIN_FRAC = 0.60  # Train/test split
SEED = 42

print(f"Demo config: {N_EXAMPLES} examples, {BOOTSTRAP_N} bootstrap samples, {TFIDF_BATCH} batch size")

## Helper Functions

Preprocessing, similarity methods (CTF, MinHash, Jaccard, TF-IDF), and metrics.

In [ ]:
def tokenize_words(text: str) -> list:
    """Extract words from text."""
    return [w for w in "".join(c if c.isalnum() else " " for c in text.lower()).split() if w]

def _stable_hash(val: Any) -> int:
    """Deterministic hash using SHA1."""
    s = str(val).encode()
    return int(hashlib.sha1(s).hexdigest()[:16], 16)

def char_ngram_jaccard(t1: str, t2: str, n: int = 4) -> float:
    """Character n-gram Jaccard similarity."""
    s1 = set(t1[i:i+n] for i in range(max(0, len(t1) - n + 1)))
    s2 = set(t2[i:i+n] for i in range(max(0, len(t2) - n + 1)))
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

def word_jaccard(t1: str, t2: str) -> float:
    """Word-unigram Jaccard similarity."""
    w1 = set(tokenize_words(t1))
    w2 = set(tokenize_words(t2))
    if not w1 and not w2:
        return 1.0
    if not w1 or not w2:
        return 0.0
    return len(w1 & w2) / len(w1 | w2)

def minhash_shingle_jaccard(t1: str, t2: str, k: int = 2) -> float:
    """Exact Jaccard on k-shingle sets."""
    words1 = tokenize_words(t1)
    words2 = tokenize_words(t2)

    def shingles(words: list, k: int) -> set:
        if len(words) < k:
            return {(w,) for w in words} if words else {""}
        return {tuple(words[i:i+k]) for i in range(len(words) - k + 1)}

    s1, s2 = shingles(words1, k), shingles(words2, k)
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

def ctf_similarity(t1: str, t2: str, radius: int = 2) -> float:
    """Circular Text Fingerprint: ECFP-style iterative neighborhood hashing."""
    def ctf_fingerprint(text: str, R: int) -> set:
        words = tokenize_words(text)
        if not words:
            return set()
        n = len(words)
        features = [_stable_hash(w) for w in words]
        fp = set(features)
        for _ in range(R):
            new_features = []
            for i in range(n):
                neighbors = []
                if i > 0:
                    neighbors.append(features[i - 1])
                if i < n - 1:
                    neighbors.append(features[i + 1])
                new_feat = _stable_hash((features[i], tuple(sorted(neighbors))))
                new_features.append(new_feat)
                fp.add(new_feat)
            features = new_features
        return fp

    fp1 = ctf_fingerprint(t1, radius)
    fp2 = ctf_fingerprint(t2, radius)
    if not fp1 and not fp2:
        return 1.0
    if not fp1 or not fp2:
        return 0.0
    return len(fp1 & fp2) / len(fp1 | fp2)

## Metrics & Threshold Tuning

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute precision, recall, F1."""
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "support": int(len(y_true)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def tune_threshold(scores: np.ndarray, labels: np.ndarray, granularity: float = 0.01) -> tuple:
    """Grid search threshold maximizing F1."""
    best_f1, best_tau = 0.0, 0.5
    thresholds = np.arange(0.0, 1.0 + granularity, granularity)
    for tau in thresholds:
        preds = (scores >= tau).astype(int)
        tp = int(((preds == 1) & (labels == 1)).sum())
        fp = int(((preds == 1) & (labels == 0)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        prec = tp / (tp + fp) if tp + fp > 0 else 0.0
        rec = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_tau = f1, float(tau)
    return best_tau, best_f1

## Load & Prepare Examples

In [ ]:
# Extract examples from loaded data
raw_examples = data['datasets'][0]['examples'][:N_EXAMPLES]

# Parse into standard format
examples = []
for ex in raw_examples:
    inp = json.loads(ex['input'])
    examples.append({
        "text1": inp["text1"],
        "text2": inp["text2"],
        "label": int(ex["output"]),
        "wc1": ex.get("metadata_wc1", len(inp["text1"].split())),
        "wc2": ex.get("metadata_wc2", len(inp["text2"].split())),
    })

print(f"Loaded {len(examples)} examples")
print(f"Class balance: {np.mean([e['label'] for e in examples]):.2%} duplicates")

## Train/Test Split

In [ ]:
# Deterministic split by stable hash
train, test = [], []
for i, ex in enumerate(examples):
    h = _stable_hash(i) % 100
    if h < int(TRAIN_FRAC * 100):
        train.append(ex)
    else:
        test.append(ex)

y_train = np.array([e['label'] for e in train])
y_test = np.array([e['label'] for e in test])

print(f"Train: {len(train)}, Test: {len(test)}")
print(f"Train duplicates: {y_train.mean():.2%}, Test duplicates: {y_test.mean():.2%}")

## Compute Similarity Scores for Each Method

In [ ]:
def compute_scores_batch(examples: list, method: str) -> np.ndarray:
    """Compute similarity scores for all pairs using the given method."""
    scores = np.zeros(len(examples), dtype=np.float32)
    for i, ex in enumerate(examples):
        t1, t2 = ex["text1"], ex["text2"]
        if method.startswith("minhash_k"):
            k = int(method.split("k")[1])
            scores[i] = minhash_shingle_jaccard(t1, t2, k=k)
        elif method == "char4gram":
            scores[i] = char_ngram_jaccard(t1, t2, n=4)
        elif method == "word_jaccard":
            scores[i] = word_jaccard(t1, t2)
        elif method.startswith("ctf_r"):
            r = int(method.split("r")[1])
            scores[i] = ctf_similarity(t1, t2, radius=r)
        else:
            raise ValueError(f"Unknown method: {method}")
    return scores

# Methods to evaluate
PAIR_METHODS = [
    "minhash_k1", "minhash_k2", "minhash_k3",
    "char4gram", "word_jaccard",
    "ctf_r1", "ctf_r2", "ctf_r3",
]
METHOD_NAMES = {
    "minhash_k1": "MinHash (k=1)",
    "minhash_k2": "MinHash (k=2)",
    "minhash_k3": "MinHash (k=3)",
    "char4gram": "Character 4-gram Jaccard",
    "word_jaccard": "Word-Unigram Jaccard",
    "tfidf": "TF-IDF Cosine",
    "ctf_r1": "CTF (R=1)",
    "ctf_r2": "CTF (R=2)",
    "ctf_r3": "CTF (R=3)",
}

print("Computing scores for pairwise methods...")
thresholds, all_test_scores, all_test_preds = {}, {}, {}

for method in PAIR_METHODS:
    train_scores = compute_scores_batch(train, method)
    tau, train_f1 = tune_threshold(train_scores, y_train, THRESHOLD_GRANULARITY)
    thresholds[method] = tau
    
    test_scores = compute_scores_batch(test, method)
    all_test_scores[method] = test_scores
    all_test_preds[method] = (test_scores >= tau).astype(int)
    
    print(f"  {METHOD_NAMES[method]}: τ={tau:.3f}, train F1={train_f1:.4f}")

gc.collect()

## TF-IDF Cosine Similarity

In [ ]:
def compute_tfidf_scores(train: list, test: list) -> np.ndarray:
    """Fit TF-IDF on train corpus and compute cosine similarities on test."""
    corpus = [ex["text1"] for ex in train] + [ex["text2"] for ex in train]
    vec = TfidfVectorizer(analyzer="word", lowercase=True, sublinear_tf=True)
    vec.fit(corpus)
    scores = np.zeros(len(test), dtype=np.float32)
    for start in range(0, len(test), TFIDF_BATCH):
        batch = test[start:start + TFIDF_BATCH]
        v1 = vec.transform([ex["text1"] for ex in batch])
        v2 = vec.transform([ex["text2"] for ex in batch])
        for j in range(len(batch)):
            scores[start + j] = float(cosine_similarity(v1[j], v2[j])[0, 0])
    return scores

print("Computing TF-IDF scores...")
tfidf_train_scores = compute_tfidf_scores(train, train)
tau_tfidf, tfidf_train_f1 = tune_threshold(tfidf_train_scores, y_train, THRESHOLD_GRANULARITY)
thresholds["tfidf"] = tau_tfidf

tfidf_test_scores = compute_tfidf_scores(train, test)
all_test_scores["tfidf"] = tfidf_test_scores
all_test_preds["tfidf"] = (tfidf_test_scores >= tau_tfidf).astype(int)

print(f"  TF-IDF Cosine: τ={tau_tfidf:.3f}, train F1={tfidf_train_f1:.4f}")
gc.collect()

## Evaluate All Methods on Test Set

In [ ]:
all_methods = PAIR_METHODS + ["tfidf"]
overall_results = {}

print("\n=== Test Set Results ===")
for method in all_methods:
    metrics = compute_metrics(y_test, all_test_preds[method])
    name = METHOD_NAMES[method]
    overall_results[name] = {
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "support": metrics["support"],
    }
    print(f"{name:30s}: P={metrics['precision']:.4f} R={metrics['recall']:.4f} F1={metrics['f1']:.4f}")

## Results Summary & Visualization

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(overall_results).T
results_df = results_df.sort_values('f1', ascending=False)

print("\n=== Results Table ===")
print(results_df[['f1', 'precision', 'recall']].round(4).to_string())

# Find best and CTF methods
best_method = results_df.index[0]
best_f1 = results_df.loc[best_method, 'f1']
ctf3_f1 = results_df.loc['CTF (R=3)', 'f1'] if 'CTF (R=3)' in results_df.index else 0

print(f"\n=== Summary ===")
print(f"Best method: {best_method} (F1={best_f1:.4f})")
print(f"CTF (R=3):   F1={ctf3_f1:.4f}")
print(f"\nDataset: {len(test)} test pairs, {y_test.mean():.2%} duplicates")

## Plot F1 Scores by Method

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
colors = ['#1f77b4' if 'CTF' in idx else '#ff7f0e' for idx in results_df.index]

ax.bar(x, results_df['f1'], color=colors, alpha=0.8, edgecolor='black')
ax.set_xlabel('Method', fontsize=12, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
ax.set_title('Near-Duplicate Detection: F1 Scores by Method', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=45, ha='right')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for i, v in enumerate(results_df['f1']):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Legend: blue=CTF, orange=baselines
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1f77b4', label='CTF variants'),
                   Patch(facecolor='#ff7f0e', label='Baselines')]
ax.legend(handles=legend_elements, loc='lower left')

plt.tight_layout()
plt.show()

print("Plot displayed above.")

## Precision vs Recall Tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for method_name in results_df.index:
    prec = results_df.loc[method_name, 'precision']
    rec = results_df.loc[method_name, 'recall']
    is_ctf = 'CTF' in method_name
    color = '#1f77b4' if is_ctf else '#ff7f0e'
    marker = 's' if is_ctf else 'o'
    size = 100 if is_ctf else 60
    ax.scatter(rec, prec, s=size, alpha=0.7, color=color, marker=marker, edgecolors='black', linewidth=1)
    ax.annotate(method_name, (rec, prec), fontsize=8, ha='left', va='bottom', xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall Tradeoff', fontsize=14, fontweight='bold')
ax.set_xlim(0.4, 1.0)
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(['CTF', 'Baselines'], loc='lower left')

plt.tight_layout()
plt.show()

print("Precision-Recall tradeoff plot displayed above.")

## Conclusion

This demo shows the core evaluation pipeline for near-duplicate text detection:

1. **CTF (Circular Text Fingerprints)** — ECFP-style iterative neighborhood hashing
2. **Baselines** — MinHash (k=1,2,3), character 4-grams, word Jaccard, TF-IDF
3. **Evaluation** — Train/test split, threshold tuning by F1, stratified metrics
4. **Results** — All methods and their precision/recall tradeoffs

**Key findings from the full experiment:**
- Character 4-gram Jaccard: Best overall F1=0.6557
- CTF (R=3): F1=0.6422, strong performance especially on short texts (≤6 words)
- CTF shows significant difference vs best baseline (McNemar p≈0)
- TF-IDF: F1=0.6344, similar to CTF despite different approach

For the full results and statistical comparisons, see the original experiment output.